## Format new data two ways, and scale it for the trained ensemble

Produces two aligned versions of the new data:
- depmap_aligned: reindexed to DepMap's full gene set and column order,
  useful for any DepMap-adjacent comparison beyond just BioBombe
- model_ready_scaled: reindexed to exactly the genes and order the trained
  ensemble expects, then scaled with the persisted MinMaxScaler

The scaler is fit on the original DepMap training data the first time this
runs, and persisted to results/depmap_gene_scaler.joblib so every later run
(on this dataset or a future one) reuses the same scale rather than
rescaling relative to whatever new cohort happens to be loaded.

Run this script with `8.apply-biobombe-new-data/` as the working directory.

In [1]:
import pathlib
import sys

import pandas as pd

sys.path.insert(0, "utils")
import data_prep as dp

In [2]:
data_directory = pathlib.Path("../0.data-download/data").resolve()
train_test_data_directory = pathlib.Path("../1.data-exploration/data").resolve()
model_save_dir = pathlib.Path("../3.run-biobombe/saved_models").resolve()
NF1_data_path = pathlib.Path("data/largaespada/NF1_data.parquet")

# The scaler is derived only from DepMap's already-public training data, not
# from the new dataset, so it's the one artifact in this module that's committed.
scaler_path = pathlib.Path("results/depmap_gene_scaler.joblib")

data_results_dir = pathlib.Path("data/largaespada/results")
data_results_dir.mkdir(parents=True, exist_ok=True)

In [3]:
prepared = dp.prepare_new_data_for_biobombe(
    new_data_path=NF1_data_path,
    data_directory=data_directory,
    train_test_data_directory=train_test_data_directory,
    model_save_dir=model_save_dir,
    scaler_path=scaler_path,
)

overlap_report = prepared["overlap_report"]
depmap_aligned = prepared["depmap_aligned"]
model_ready_scaled = prepared["model_ready_scaled"]

print(f"DepMap-aligned: {depmap_aligned.shape}")
print(f"Model-ready, scaled: {model_ready_scaled.shape}")

/home/gway/miniconda3/envs/gene_dependency_representations/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.6.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


DepMap-aligned: (8, 18444)
Model-ready, scaled: (8, 2719)


In [4]:
n_imputed_per_gene = pd.Series(model_ready_scaled.attrs["n_imputed_per_gene"])
n_genes_needing_imputation = (n_imputed_per_gene > 0).sum()
print(f"{n_genes_needing_imputation} / {len(n_imputed_per_gene)} trained genes needed imputation "
      "(missing in the new data, filled with the DepMap training mean before scaling)")

if n_genes_needing_imputation:
    print(n_imputed_per_gene.sort_values(ascending=False).head(20))

12 / 2718 trained genes needed imputation (missing in the new data, filled with the DepMap training mean before scaling)
TXNL4A (10907)     8
PPP2R3C (55012)    8
NEPRO (25871)      8
TAB2 (23118)       8
CARS2 (79587)      8
MED20 (9477)       8
CDK8 (1024)        8
SGO1 (151648)      8
CALM2 (805)        8
COQ4 (51117)       8
KAT6A (7994)       8
SPRTN (83932)      8
PSMB3 (5691)       0
PSMB1 (5689)       0
PSMB4 (5692)       0
PSMB5 (5693)       0
PSMB6 (5694)       0
PSMD13 (5719)      0
PSMB7 (5695)       0
PSMC1 (5700)       0
dtype: int64


In [5]:
depmap_aligned.to_parquet(data_results_dir / "NF1_data_depmap_aligned.parquet", index=False)
model_ready_scaled.to_parquet(data_results_dir / "NF1_data_model_ready_scaled.parquet", index=False)

print(f"Saved formatted data to {data_results_dir}")
print(f"Scaler bundle available at {scaler_path}")

Saved formatted data to data/largaespada/results
Scaler bundle available at results/depmap_gene_scaler.joblib
